# Gene Expression Correlation AnalysisThis notebook analyzes correlation patterns between predicted proteinexpression levels across cells using ProtVL predictions.**Workflow:**1. Generate protein predictions across multiple cells with ProtVL2. Save predictions to disk3. Compute mean expression per gene per cell4. Calculate pairwise correlations and hierarchical clustering5. Visualize as heatmaps and network graphs

## Setup

In [ ]:
import osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom tifffile import imread, imwritefrom skimage.filters import threshold_otsufrom tqdm.auto import tqdmfrom scipy.cluster.hierarchy import dendrogram, linkage, leaves_listfrom scipy.spatial.distance import squareformimport networkx as nxfrom protvl import ProtVL

## Configuration

In [ ]:
# Analysis parametersTARGET_GENE = "RELA"CELL_LINE = "A-431"# NF-kB pathway genes to analyzegene_names = [    "MAP3K8", "BCL3", "BRD4", "NUTM1", "RELB", "HDAC1", "NFKBIB", "NFKB2",    "REL", "EP300", "CITED4", "NFKBIE", "NFKBIA", "NFKB1", "CREBBP", "EGR3",    "NFKBIZ", "NFKBID", "KLF2", "GREB1", TARGET_GENE,]# Correlation thresholds for network visualizationPOS_THRESHOLD = 0.5NEG_THRESHOLD = 0.5# DirectoriesINPUT_IMAGE_DIR = f"examples/{TARGET_GENE}_{CELL_LINE}"PRED_IMAGE_DIR = f"{TARGET_GENE}_{CELL_LINE}_output"RESULT_DIR = f"{TARGET_GENE}_results"os.makedirs(PRED_IMAGE_DIR, exist_ok=True)os.makedirs(RESULT_DIR, exist_ok=True)print(f"Analyzing {len(gene_names)} genes in {CELL_LINE}")

## Helper Functions

In [ ]:
def calculate_mean_expression(image, thresh_multiplier=1.25):    """Calculate mean expression in thresholded regions."""    thresh = thresh_multiplier * threshold_otsu(image)    binary_mask = image > thresh    if np.any(binary_mask):        return np.mean(image[binary_mask])    return 0.0def zscore_normalize(df, gene_cols):    """Z-score normalize each gene across cells."""    df_norm = df.copy()    for col in gene_cols:        vals = df[col].values        finite_vals = vals[np.isfinite(vals)]        if len(finite_vals) > 0:            mean_val = np.mean(finite_vals)            std_val = np.std(finite_vals)            df_norm[col] = (df[col] - mean_val) / (std_val + 1e-8)    return df_normdef reorder_by_clustering(corr_matrix, gene_list):    """Reorder correlation matrix using hierarchical clustering."""    distance_matrix = 1 - corr_matrix    np.fill_diagonal(distance_matrix, 0)    condensed_dist = squareform(distance_matrix, checks=False)    linkage_matrix = linkage(condensed_dist, method="average")    order = leaves_list(linkage_matrix)    reordered_matrix = corr_matrix[order, :][:, order]    reordered_genes = [gene_list[i] for i in order]    return reordered_matrix, reordered_genes, linkage_matrix

## Step 1: Generate Predictions with ProtVLPredict each gene across all reference cells and save to disk.

In [ ]:
model = ProtVL()ref_files = sorted([f for f in os.listdir(INPUT_IMAGE_DIR) if f.endswith(".tiff")])N_CELLS = len(ref_files)print(f"Found {N_CELLS} reference cells")# Predict all non-target genes for each cellgenes_to_predict = [g for g in gene_names if g != TARGET_GENE]for cell_idx, cell_file in enumerate(tqdm(ref_files, desc="Cells")):    ref_image = imread(os.path.join(INPUT_IMAGE_DIR, cell_file))    for gene in tqdm(genes_to_predict, desc=f"  Cell {cell_idx}", leave=False):        out_path = os.path.join(PRED_IMAGE_DIR, f"cell_{cell_idx}_{CELL_LINE}_{gene}_pred.tif")        if os.path.exists(out_path):            continue        result = model.predict(            images=[ref_image],            protein_names=[gene],            cell_line_names=[CELL_LINE],            num_inference_steps=50,        )        imwrite(out_path, result[0].astype(np.float32))del modelimport torch; torch.cuda.empty_cache()print(f"Predictions saved to {PRED_IMAGE_DIR}/")

## Step 2: Calculate Mean Expression Per Cell

In [ ]:
cell_data = []for i in tqdm(range(N_CELLS), desc=f"Processing {CELL_LINE}"):    cell_expr = {"cell_id": f"{CELL_LINE}_cell_{i}"}    # Load target gene from real image    real_path = os.path.join(INPUT_IMAGE_DIR, ref_files[i])    real_image = imread(real_path)    if real_image.ndim == 3 and real_image.shape[-1] >= 2:        real_protein = real_image[:, :, 1].astype(np.float32)        real_protein = (real_protein + 1) / 2  # normalize        cell_expr[TARGET_GENE] = calculate_mean_expression(real_protein)    # Load predicted images for other genes    for gene in genes_to_predict:        pred_path = os.path.join(PRED_IMAGE_DIR, f"cell_{i}_{CELL_LINE}_{gene}_pred.tif")        if not os.path.exists(pred_path):            continue        pred_image = imread(pred_path).astype(np.float32)        cell_expr[gene] = calculate_mean_expression(pred_image)    cell_data.append(cell_expr)expression_df = pd.DataFrame(cell_data).set_index("cell_id")expression_df.to_csv(f"{RESULT_DIR}/{CELL_LINE}_expression.csv")print(f"Processed {len(expression_df)} cells, {len(expression_df.columns)} genes")

## Step 3: Correlation Analysis

In [ ]:
# Z-score normalizeavailable_genes = [g for g in gene_names if g in expression_df.columns]expression_zscore = zscore_normalize(expression_df, available_genes)expression_zscore.to_csv(f"{RESULT_DIR}/{CELL_LINE}_expression_zscore.csv")# Correlation matrixdata = expression_zscore[available_genes].valuesmask = np.all(np.isfinite(data), axis=1)corr_matrix = np.corrcoef(data[mask].T)corr_df = pd.DataFrame(corr_matrix, index=available_genes, columns=available_genes)corr_df.to_csv(f"{RESULT_DIR}/{CELL_LINE}_correlation_matrix.csv")print(f"Correlation matrix: {corr_matrix.shape}")

## Step 4: Target Gene Correlations

In [ ]:
if TARGET_GENE in available_genes:    target_idx = available_genes.index(TARGET_GENE)    other_genes = [g for i, g in enumerate(available_genes) if i != target_idx]    other_corrs = [corr_matrix[target_idx, i] for i in range(len(available_genes)) if i != target_idx]    sorted_idx = np.argsort(other_corrs)    genes_sorted = [other_genes[i] for i in sorted_idx]    corr_sorted = [other_corrs[i] for i in sorted_idx]    colors = ["#d62728" if c < 0 else "#1f77b4" for c in corr_sorted]    fig, ax = plt.subplots(figsize=(10, 8))    y_pos = np.arange(len(genes_sorted))    bars = ax.barh(y_pos, corr_sorted, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)    ax.set_yticks(y_pos)    ax.set_yticklabels(genes_sorted, fontsize=11, fontweight="bold")    ax.set_xlabel("Correlation Coefficient", fontsize=12, fontweight="bold")    ax.set_title(f"{TARGET_GENE} Correlations ({CELL_LINE})", fontsize=14, fontweight="bold", pad=20)    ax.axvline(x=0, color="black", linestyle="-", linewidth=1.2, alpha=0.7)    ax.grid(axis="x", alpha=0.3, linestyle="--")    for bar, val in zip(bars, corr_sorted):        x_pos = val + (0.02 if val > 0 else -0.02)        ha = "left" if val > 0 else "right"        ax.text(x_pos, bar.get_y() + bar.get_height()/2, f"{val:.3f}",                va="center", ha=ha, fontsize=9, fontweight="bold")    from matplotlib.patches import Patch    ax.legend(handles=[        Patch(facecolor="#1f77b4", alpha=0.8, label="Positive"),        Patch(facecolor="#d62728", alpha=0.8, label="Negative"),    ], loc="lower right", fontsize=10)    plt.tight_layout()    plt.savefig(f"{RESULT_DIR}/{CELL_LINE}_{TARGET_GENE}_correlations.png", dpi=300, bbox_inches="tight")    plt.show()    print(f"\nTop 3 positive: {', '.join(f'{genes_sorted[-(i+1)]} ({corr_sorted[-(i+1)]:.3f})' for i in range(min(3, len(genes_sorted))))}")    print(f"Top 3 negative: {', '.join(f'{genes_sorted[i]} ({corr_sorted[i]:.3f})' for i in range(min(3, len(genes_sorted))))}")

## Step 5: Hierarchical Clustering & Heatmap

In [ ]:
reordered_corr, reordered_genes, linkage_matrix = reorder_by_clustering(    corr_matrix, available_genes)fig = plt.figure(figsize=(14, 12))gs = fig.add_gridspec(2, 2, width_ratios=[0.2, 1], height_ratios=[0.2, 1],                      hspace=0.02, wspace=0.02)ax_top = fig.add_subplot(gs[0, 1])dendrogram(linkage_matrix, ax=ax_top, orientation="top", no_labels=True,           color_threshold=0, above_threshold_color="k")ax_top.axis("off")ax_left = fig.add_subplot(gs[1, 0])dendrogram(linkage_matrix, ax=ax_left, orientation="left", no_labels=True,           color_threshold=0, above_threshold_color="k")ax_left.axis("off")ax = fig.add_subplot(gs[1, 1])im = ax.imshow(reordered_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)cbar.set_label("Correlation", rotation=270, labelpad=20)ax.set_xticks(np.arange(len(reordered_genes)) - 0.5, minor=True)ax.set_yticks(np.arange(len(reordered_genes)) - 0.5, minor=True)ax.grid(which="minor", color="lightgray", linestyle="-", linewidth=0.5)ax.tick_params(which="minor", size=0)ax.set_xticks(range(len(reordered_genes)))ax.set_yticks(range(len(reordered_genes)))ax.set_xticklabels(reordered_genes, rotation=45, ha="right", fontsize=8)ax.set_yticklabels(reordered_genes, fontsize=8)for i in range(len(reordered_genes)):    for j in range(len(reordered_genes)):        if abs(reordered_corr[i, j]) > 0.5:            color = "white" if abs(reordered_corr[i, j]) > 0.7 else "black"            ax.text(j, i, f"{reordered_corr[i, j]:.2f}",                   ha="center", va="center", color=color, fontsize=6)plt.suptitle(f"Gene Correlation Matrix - {CELL_LINE}", fontsize=14, fontweight="bold")plt.savefig(f"{RESULT_DIR}/{CELL_LINE}_correlation_heatmap.png", dpi=300, bbox_inches="tight")plt.show()

## Step 6: Network Graph

In [ ]:
G = nx.Graph()G.add_nodes_from(reordered_genes)positive_edges, negative_edges = [], []pos_weights, neg_weights = [], []for i in range(len(reordered_genes)):    for j in range(i + 1, len(reordered_genes)):        corr = reordered_corr[i, j]        edge = (reordered_genes[i], reordered_genes[j])        if corr >= POS_THRESHOLD:            G.add_edge(*edge, weight=corr, edge_type="positive")            positive_edges.append(edge)            pos_weights.append(corr)        elif corr <= -NEG_THRESHOLD:            G.add_edge(*edge, weight=abs(corr), edge_type="negative")            negative_edges.append(edge)            neg_weights.append(abs(corr))fig, ax = plt.subplots(figsize=(14, 14))pos = nx.spring_layout(G, k=2, iterations=50, seed=42)nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=800,                      edgecolors="darkblue", linewidths=2, ax=ax)if positive_edges:    nx.draw_networkx_edges(G, pos, edgelist=positive_edges,                          edge_color="green", width=[w * 3 for w in pos_weights],                          alpha=0.7, style="solid", ax=ax)if negative_edges:    nx.draw_networkx_edges(G, pos, edgelist=negative_edges,                          edge_color="red", width=[w * 3 for w in neg_weights],                          alpha=0.7, style="dashed", ax=ax)nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold", ax=ax)from matplotlib.lines import Line2Dax.legend(handles=[    Line2D([0], [0], color="green", linewidth=2, linestyle="solid",           label=f"Positive (n={len(positive_edges)})"),    Line2D([0], [0], color="red", linewidth=2, linestyle="dashed",           label=f"Negative (n={len(negative_edges)})"),], loc="upper right", fontsize=11)ax.set_title(f"Gene Correlation Network - {CELL_LINE}", fontsize=14, fontweight="bold")ax.axis("off")plt.tight_layout()plt.savefig(f"{RESULT_DIR}/{CELL_LINE}_correlation_network.png", dpi=300, bbox_inches="tight")plt.show()nx.write_gml(G, f"{RESULT_DIR}/{CELL_LINE}_correlation_network.gml")print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Summary**Outputs:**- `{CELL_LINE}_expression.csv` — Raw mean expression per cell- `{CELL_LINE}_expression_zscore.csv` — Z-score normalized- `{CELL_LINE}_correlation_matrix.csv` — Pairwise correlations- `{CELL_LINE}_{TARGET_GENE}_correlations.png` — Bar chart- `{CELL_LINE}_correlation_heatmap.png` — Clustered heatmap- `{CELL_LINE}_correlation_network.png` — Network graph- `{CELL_LINE}_correlation_network.gml` — Network file